In [ ]:
import usb.core
import usb.util
import struct
import numpy as np
import matplotlib.pyplot as plt
import time
import pandas as pd
from datetime import datetime
from scipy.signal import savgol_filter
from IPython.display import clear_output

In [ ]:
class Aris546ZioDriver:
    def __init__(self):
        self.dev = usb.core.find(idVendor=0x354F)
        if self.dev is None:
            raise ConnectionError("Aris 546 not found.")
        self.dev.set_configuration()
        self.pixels = 2048

    def get_spectrum(self, exp_ms=2.0):
        try:
            self.dev.write(0x01, struct.pack('<II', 0x1100, int(exp_ms * 1000)))
            self.dev.write(0x01, struct.pack('<II', 0x0004, 1))

            for _ in range(50):
                self.dev.write(0x01, struct.pack('<I', 0x3000) + b'\x00'*4)
                reply = self.dev.read(0x81, 16, timeout=500)
                if len(reply) >= 8:
                    status_val = struct.unpack('<I', reply[4:8])[0]
                    if (status_val >> 8) & 0xFFFF > 0:
                        break
                time.sleep(0.01)

            self.dev.write(0x01, struct.pack('<I', 0x4000) + b'\x00'*4)
            data_reply = self.dev.read(0x81, 16492, timeout=2000)

            return np.frombuffer(data_reply[68:68 + (2048 * 4)], dtype=np.float32)

        except:
            return None

In [ ]:
wavelengths = np.linspace(182.98, 988.74, 2048)

dark = None
ref = None
sample = None

hw = Aris546ZioDriver()
print("Spectrometer Connected")

In [ ]:
def capture_dark():
    global dark
    dark = hw.get_spectrum()
    print("Dark captured")

def capture_ref():
    global ref
    ref = hw.get_spectrum()
    print("Reference captured")

def capture_sample():
    global sample
    sample = hw.get_spectrum()
    print("Sample captured")

In [ ]:
def compute_absorbance(spectrum):
    global dark, ref

    ratio = (spectrum - dark) / (ref - dark + 1e-7)
    absorbance = -np.log10(np.clip(ratio, 0.001, 2.0))
    absorbance = savgol_filter(absorbance, 21, 3)

    return absorbance

In [ ]:
def plot_live(absorbance):
    plt.figure(figsize=(8,4))
    plt.plot(wavelengths, absorbance)
    plt.title("Live Absorbance")
    plt.xlabel("Wavelength (nm)")
    plt.ylabel("Absorbance")
    plt.grid()
    plt.show()

In [ ]:
def plot_raw():
    plt.figure(figsize=(8,4))

    if dark is not None:
        plt.plot(wavelengths, dark, label="Dark")
    if ref is not None:
        plt.plot(wavelengths, ref, label="Reference")
    if sample is not None:
        plt.plot(wavelengths, sample, label="Sample")

    plt.legend()
    plt.title("Raw Spectra Comparison")
    plt.show()

In [ ]:
def plot_uv(absorbance):
    mask = (wavelengths >= 200) & (wavelengths <= 350)

    plt.figure(figsize=(8,4))
    plt.plot(wavelengths[mask], absorbance[mask])
    plt.title("UV Region (200–350 nm)")
    plt.xlabel("Wavelength (nm)")
    plt.ylabel("Absorbance")
    plt.show()

In [ ]:
def estimate_nitrate(absorbance):
    ppm = max(0, absorbance[150] * 10.5)
    print(f"Nitrate Concentration: {ppm:.2f} ppm")
    return ppm

In [ ]:
def plot_result(ppm):
    plt.figure(figsize=(4,4))
    plt.bar(["Nitrate"], [ppm])
    plt.title(f"Concentration: {ppm:.2f} ppm")
    plt.show()

In [ ]:
capture_dark()
time.sleep(1)

capture_ref()
time.sleep(1)

capture_sample()

abs_data = compute_absorbance(sample)

plot_live(abs_data)
plot_raw()
plot_uv(abs_data)

ppm = estimate_nitrate(abs_data)
plot_result(ppm)